# 5. RAG: answer questions from rulings

Ask a question in English. The system finds the most relevant passages in a set of Dutch rulings and lets Gemini answer from those passages only, citing each ruling by ECLI.

Pipeline (code in `../rag/`):

1. Translate the question to Dutch (Gemini API).
2. Vector search with `multilingual-e5-base`: 30 candidate passages from Chroma.
3. Rerank each (question, passage) pair with a cross-encoder. Keep passages with probability at least 0.5, at most 5, at most 2 per ruling.
4. Gemini answers from the kept passages, with the section of each passage (court's assessment, party argument, ...) in the prompt.
5. Refuse if no passage is kept, or if the draft answer says the passages do not address the question.

Nothing is trained here. All models are used as they are.

Before running: `rag/build_corpus.py`, `rag/chunk.py` and `rag/build_index.py` have been run once, and `python -m rag.evaluate` has written `../results/rag_answers.jsonl`. This notebook makes no Gemini calls: it reads the Dutch queries and answers recorded by the evaluation and runs retrieval locally. To ask a new question, use `python -m rag.answer "..."`.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config
from rag.retrieve import Retriever

for path in [config.RAG_CORPUS, config.RAG_CHUNKS, config.RAG_INDEX_DIR]:
    print(path.relative_to(ROOT), "exists:", path.exists())

summary = json.loads((ROOT / "results" / "rag_eval.json").read_text(encoding="utf-8"))
records = pd.read_json(ROOT / "results" / "rag_answers.jsonl", lines=True)
by_id = records.set_index("id")

C:\Users\nlwen\Desktop\hbo-corp-justid\justid-machinelearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data\rag\corpus.jsonl exists: True
data\rag\chunks.jsonl exists: True
data\rag\chroma exists: True


## Corpus and chunks

2 500 rulings from `test` and `natural_test`. Twelve topics are defined by a keyword rule plus a rechtsgebied (see `build_corpus.py`). The rest are random rulings with no topic, so retrieval has something to get wrong. Rulings that mention a refusal subject (flight delays, crypto, ...) were left out on purpose.

In [2]:
corpus = pd.read_json(config.RAG_CORPUS, lines=True)
print("rulings:", len(corpus))
print("no topic (distractors):", int((corpus["topics"].map(len) == 0).sum()))
corpus["topics"].explode().value_counts()

rulings: 2500
no topic (distractors): 1618


topics
environmental_permit    100
bijstand_recovery       100
tenancy                 100
tbs                     100
debt_restructuring       81
wia                      75
contact_arrangement      70
woz_value                61
child_support            59
asylum_credibility       51
drink_driving            49
summary_dismissal        43
Name: count, dtype: int64

Each ruling is cut into passages of about 250 words with 40 words of overlap. A passage never crosses a section heading, so every passage has one section label.

In [3]:
chunks = pd.read_json(config.RAG_CHUNKS, lines=True)
print("chunks:", len(chunks))
print("words per chunk, median:", int(chunks["text"].str.split().map(len).median()))
chunks["section"].value_counts(normalize=True).round(3)

chunks: 44490


words per chunk, median: 200


section
court        0.519
procedure    0.143
parties      0.104
decision     0.079
facts        0.072
header       0.063
advisory     0.019
Name: proportion, dtype: float64

## Retrieval in two stages

Vector scores from e5 sit in a narrow band (about 0.85 to 0.89) for relevant and irrelevant passages alike, so they cannot decide what to keep. The cross-encoder reads question and passage together and gives a probability that separates them much better.

In [4]:
retriever = Retriever()

rec = by_id.loc["q09"]
question, dutch = rec["question"], rec["dutch_query"]
print("Question:", question)
print("Dutch query:", dutch)

stage1 = sorted(retriever.candidates([dutch, question], n=30), key=lambda h: h.score, reverse=True)
pd.DataFrame([{"ecli": h.ecli, "section": h.section, "vector": round(h.score, 3)} for h in stage1[:5]])

Question: How do courts decide whether the WOZ value of a home was set too high?
Dutch query: Hoe beoordeelt de rechter of de WOZ-waarde van een woning te hoog is vastgesteld?


,ecli,section,vector
0,ECLI:NL:RBMNE:2021:5143,parties,0.898
1,ECLI:NL:RBZWB:2025:1007,court,0.896
2,ECLI:NL:RBZWB:2024:6439,court,0.896
3,ECLI:NL:GHDHA:2024:2680,court,0.895
4,ECLI:NL:RBZWB:2021:4426,court,0.895


In [5]:
stage2 = retriever.search(question, dutch)
pd.DataFrame([{"ecli": h.ecli, "section": h.section, "vector": round(h.score, 3), "rerank": round(h.rerank, 3)} for h in stage2])

,ecli,section,vector,rerank
0,ECLI:NL:RBZWB:2024:6439,court,0.896,1.0
1,ECLI:NL:RBZWB:2024:300,court,0.894,1.0
2,ECLI:NL:RBZWB:2024:6332,court,0.892,1.0
3,ECLI:NL:RBZWB:2025:1007,court,0.896,1.0
4,ECLI:NL:RBZWB:2024:8603,court,0.887,1.0


## Full answer

In [6]:
print(rec["answer"])
print()
for s in rec["sources"]:
    print(f"{s['ecli']}  [{s['section']}]  rerank {s['rerank']:.2f}")
print("citations outside the passages:", rec["invalid_citations"])

The court assesses whether the value of the home was set too high based on the arguments and grounds for appeal submitted by the interested party [ECLI:NL:RBZWB:2024:6439], [ECLI:NL:RBZWB:2024:300], [ECLI:NL:RBZWB:2024:6332], [ECLI:NL:RBZWB:2025:1007], [ECLI:NL:RBZWB:2024:8603]. Under Article 17, second paragraph, of the Wet WOZ, the value is determined based on the price that the highest bidder would spend if the property were offered for sale in the most suitable manner after the best preparation, assuming full and unencumbered ownership can be transferred and the buyer can immediately and fully use the property in its current state [ECLI:NL:RBZWB:2024:300], [ECLI:NL:RBZWB:2024:6332], [ECLI:NL:RBZWB:2025:1007].

ECLI:NL:RBZWB:2024:6439  [court]  rerank 1.00
ECLI:NL:RBZWB:2024:300  [court]  rerank 1.00
ECLI:NL:RBZWB:2024:6332  [court]  rerank 1.00
ECLI:NL:RBZWB:2025:1007  [court]  rerank 1.00
ECLI:NL:RBZWB:2024:8603  [court]  rerank 1.00
citations outside the passages: []


## Refusal

Two ways to refuse. Crypto: no passage passes the reranker, so the LLM is never called. Solar panels: some passages pass (roof repairs, "solartubes" skylights), Gemini writes a draft, and the draft itself says the rulings do not cover the question.

In [7]:
for qid in ["r05", "r03"]:
    r = by_id.loc[qid]
    print(r["question"])
    print("  passages kept:", len(r["sources"]))
    print("  draft:", r["draft"][:300] if r["draft"] else "(LLM not called)")
    print("  final:", r["answer"])
    print()

How is cryptocurrency taxed in box 3?
  passages kept: 0
  draft: (LLM not called)
  final: Not found in the provided rulings.

Can a homeowner be forced to remove solar panels from their roof?
  passages kept: 4
  draft: The provided passages do not contain any information about whether a homeowner can be forced to remove solar panels from their roof.
  final: Not found in the provided rulings.



## Evaluation

`python -m rag.evaluate` runs all 29 questions in `../rag/eval/questions.jsonl` (24 topic questions, 5 that must be refused) and writes `../results/rag_eval.json` and `../results/rag_answers.jsonl`.

- **retrieval_hit_rate**: at least one kept passage comes from a ruling with the question's topic
- **retrieval_precision**: share of kept passages from on-topic rulings
- **refusal_correct**: refusal questions that were refused
- **invalid_citation_rate**: cited ECLIs that are not among the passages given to the LLM

In [8]:
pd.Series({k: v for k, v in summary.items() if k != "settings"})

questions                29.0
topical_questions        24.0
refusal_questions         5.0
retrieval_hit_rate        1.0
retrieval_precision       0.9
topical_answered          1.0
refusal_correct           1.0
answers_with_citation     1.0
invalid_citation_rate     0.0
median_seconds            4.7
dtype: float64

In [9]:
records["kept"] = records["sources"].map(len)
records["on_topic"] = records["sources"].map(lambda s: sum(bool(x["on_topic"]) for x in s))
records["cited"] = records["cited"].map(len)
records[["id", "topic", "refused", "kept", "on_topic", "cited", "seconds"]]

,id,topic,refused,kept,on_topic,cited,seconds
0,q01,tenancy,False,5,4,3,8.4
1,q02,tenancy,False,5,5,2,5.2
2,q03,summary_dismissal,False,5,5,3,4.7
3,q04,summary_dismissal,False,5,5,2,5.6
4,q05,bijstand_recovery,False,5,5,4,4.4
5,q06,bijstand_recovery,False,5,3,3,4.7
6,q07,asylum_credibility,False,5,5,3,179.6
7,q08,asylum_credibility,False,5,5,3,5.6
8,q09,woz_value,False,5,5,5,3.8
9,q10,woz_value,False,5,5,3,5.7


## Manual check

A valid ECLI does not mean a correct claim. Three answers from the evaluation run, each shown next to the passages it was given. Retrieval is deterministic, so searching again with the recorded Dutch query returns the same passages.

In [10]:
def show(qid: str) -> None:
    rec = by_id.loc[qid]
    hits = retriever.search(rec["question"], rec["dutch_query"])
    assert [h.ecli for h in hits] == [s["ecli"] for s in rec["sources"]], "passages differ from the eval run"
    print(rec["question"])
    print()
    print(rec["answer"])
    for h in hits:
        print(f"\n--- {h.ecli} [{h.section}]")
        print(h.text)

In [11]:
show("q19")

Under what conditions do courts impose TBS with compulsory treatment?

The court imposes the TBS measure with compulsory treatment when the statutory conditions are met, which include a defect development or pathological disorder of the defendant's mental faculties during the commission of the proven facts [ECLI:NL:RBMNE:2024:5671]. Additionally, the offenses committed must be crimes for which TBS can be imposed, and the safety of others or general safety requires this measure because the risk of recidivism arising from the disorder remains high without treatment [ECLI:NL:RBMNE:2024:5671]. Furthermore, a court may consider that there is no alternative treatment available due to the defendant's refusal to cooperate with personal investigations, and the seriousness of the offenses justifies the measure [ECLI:NL:RBMNE:2022:2405]. Finally, a court may find TBS with compulsory treatment necessary when less intrusive measures were investigated but deemed insufficient, and only TBS with compu

In [12]:
show("q14")

When can child support be reduced because the paying parent's income has dropped?

A judicial decision or maintenance agreement can be modified or revoked if subsequent changes in circumstances mean it no longer meets statutory standards [ECLI:NL:GHARL:2021:10432]. Furthermore, a party may argue that changed circumstances—such as altered personal circumstances, reduced earning capacity, or an inability to cover fixed costs and maintenance—warrant reducing a child support contribution [ECLI:NL:RBZWB:2024:6687].

--- ECLI:NL:GHARL:2018:10710 [court]
maand. * ingangsdatum 5.3 In hoger beroep is geen grief gericht tegen de door de rechtbank gehanteerde ingangsdatum. Ook het hof zal daarom de ingangsdatum van de eventuele wijziging van de kinderalimentatie stellen op 1 oktober 2017. * behoefte kinderen 5.4 Partijen hebben niets gesteld over de (toenmalige dan wel huidige) behoefte van de kinderen. Ter zitting is enkel in algemene bewoordingen benoemd dat aan de destijds vastgestelde kindera

In [13]:
show("q11")

When do courts restrict or refuse contact between a parent and a child?

Under Article 1:377a paragraph 3 of the Civil Code, a court denies the right to contact between a parent and a child only if contact would cause serious harm to the child's mental or physical development, the parent is manifestly unsuitable or incapable, a child of twelve years or older has raised serious objections during a hearing, or contact is otherwise contrary to the child's weighty interests [ECLI:NL:GHAMS:2022:1315, ECLI:NL:GHARL:2018:10854]. Furthermore, a court will not bring about forced contact between a child and a parent if it is not in the child's best interests, allowing the child time to take the step toward contact independently [ECLI:NL:GHARN:2011:BQ5048].

--- ECLI:NL:RBZWB:2024:6687 [court]
het daarom ook enorm moeilijk om zijn vader te treffen in de supermarkt waar hij op dat moment aan het werk was. Na het voorval in de supermarkt is de man buiten zinnen van woede naar de woning van de vrouw

### Verdict

Each claim checked against the passage it cites:

- **q19 (TBS)**: 5 of 5 correct. The three statutory conditions, the refusal to cooperate, and why TBS with conditions was not enough all match the cited rulings. ECLI:NL:RBNNE:2025:3679, where the court chose TBS with conditions, is not cited as an example of compulsory treatment.
- **q14 (child support)**: 2 of 2 correct, but incomplete. Article 1:401 BW is stated correctly. The man's claim in ECLI:NL:RBZWB:2024:6687 is presented as his argument, even though the passage carries the wrong section label (court). None of the passages says whether a reduction was granted, so the answer does not either.
- **q11 (contact)**: 2 of 2 correct. The four grounds of Article 1:377a(3) BW and the suspension of contact in ECLI:NL:GHARN:2011:BQ5048 match the passages. The party arguments in ECLI:NL:GHSHE:2015:2229 are not used.

Limits: 29 questions is a small set, and the answer prompt was adjusted after seeing false refusals on some of these questions, so they are not fully held out. A correct claim also does not mean a complete answer: the system only knows what its five passages say.